In [1]:
import json
import importlib
from dataclasses import dataclass, field
from datetime import datetime, date
from typing import Any, Dict, Optional

import fetch_retry
from fetch_retry import fetch_com_retry

In [2]:

@dataclass
class PncpApiConfig:
    """
    Responsável por armazenar a configuração base da API do PNCP.
    """
    base_url: str = "https://pncp.gov.br/api/consulta"
    endpoint: str = "/v1/contratacoes/proposta"

    @property
    def url_final(self) -> str:
        return f"{self.base_url}{self.endpoint}"


In [3]:
@dataclass
class ConsultaPropostaParams:
    """
    Responsável por representar os parâmetros da consulta.
    """
    codigo_modalidade: int = 6
    pagina: int = 1
    tamanho_pagina: int = 50
    data_final: int = field(default_factory=lambda: int(datetime.now().strftime("%Y%m%d")))

    def to_dict(self) -> Dict[str, Any]:
        return {
            "dataFinal": self.data_final,
            "codigoModalidadeContratacao": self.codigo_modalidade,
            "pagina": self.pagina,
            "tamanhoPagina": self.tamanho_pagina,
        }

In [4]:

class ModuloFetcherLoader:
    """
    Responsável por recarregar dinamicamente o módulo de fetch,
    útil em ambiente de desenvolvimento/notebook.
    """

    def __init__(self, module_name: str = "fetch_retry") -> None:
        self.module_name = module_name

    def reload(self) -> None:
        module = importlib.import_module(self.module_name)
        importlib.reload(module)
        print(f"Módulo '{self.module_name}' recarregado com sucesso.")

In [5]:
class JsonFormatter:
    """
    Responsável por formatar estruturas Python em JSON legível.
    """

    @staticmethod
    def format(data: Any) -> str:
        return json.dumps(data, indent=4, ensure_ascii=False)

In [6]:
class PncpConsultaService:
    """
    Responsável por executar a consulta na API do PNCP.
    """

    def __init__(self, api_config: PncpApiConfig) -> None:
        self.api_config = api_config

    def consultar_propostas(self, params: ConsultaPropostaParams) -> Dict[str, Any]:
        return fetch_com_retry(self.api_config.url_final, params.to_dict())


In [7]:
class PncpConsultaOrchestrator:
    """
    Responsável por orquestrar o fluxo completo:
    - recarregar módulo de fetch
    - montar parâmetros
    - executar consulta
    - exibir resultados
    """

    def __init__(
        self,
        consulta_service: PncpConsultaService,
        formatter: JsonFormatter,
        loader: Optional[ModuloFetcherLoader] = None,
    ) -> None:
        self.consulta_service = consulta_service
        self.formatter = formatter
        self.loader = loader
        self.data_hoje = date.today().strftime("%Y-%m-%d")

    def executar(self) -> Dict[str, Any]:
        if self.loader:
            self.loader.reload()

        params = ConsultaPropostaParams()

        print(f"Data final da consulta: {params.data_final}")
        print(f"Data de execução do pipeline: {self.data_hoje}")
        print(f"URL da consulta: {self.consulta_service.api_config.url_final}")
        print("Parâmetros enviados:")
        print(self.formatter.format(params.to_dict()))

        resposta = self.consulta_service.consultar_propostas(params)

        print("\nResposta bruta:")
        print(resposta)

        print("\nResposta formatada:")
        print(self.formatter.format(resposta))

        return resposta

In [8]:

if __name__ == "__main__":
    api_config = PncpApiConfig()
    consulta_service = PncpConsultaService(api_config)
    formatter = JsonFormatter()
    loader = ModuloFetcherLoader()

    orchestrator = PncpConsultaOrchestrator(
        consulta_service=consulta_service,
        formatter=formatter,
        loader=loader,
    )

    resultado = orchestrator.executar()

Módulo 'fetch_retry' recarregado com sucesso.
Data final da consulta: 20260319
Data de execução do pipeline: 2026-03-19
URL da consulta: https://pncp.gov.br/api/consulta/v1/contratacoes/proposta
Parâmetros enviados:
{
    "dataFinal": 20260319,
    "codigoModalidadeContratacao": 6,
    "pagina": 1,
    "tamanhoPagina": 50
}
Código de resposta 200 (OK)

Resposta bruta:
([{'dataAtualizacao': '2025-10-14T10:08:45', 'orgaoEntidade': {'cnpj': '30066661000180', 'razaoSocial': 'SUPERINTENDENCIA MUNICIPAL DE TRANSITO', 'poderId': 'E', 'esferaId': 'M'}, 'anoCompra': 2025, 'sequencialCompra': 15, 'numeroCompra': '43248', 'processo': '2025017256', 'objetoCompra': 'ADESÃO Nº 038/2025\nADESÃO ATA DE REGISTRO DE PREÇOS Nº 067/2025 DE 19/03/2025 EM DECORRÊNCIA PREGÃO ELETRONICO/SRP Nº 004/2025 - PROCESSO ADMINISTRATIVO Nº 2024013537 DA PREFEITURA MUNICIPAL DE CALDAS NOVAS/GO, NA CONDIÇÃO DE ?CARONA?, PARA EVENTUAL AQUISIÇÃO DE MATERIAL DE CONSTRUÇÃO EM ATENDIMENTO ÀS NECESSIDADES DA SUPERINTENDÊNCIA 

In [51]:
import json
import os
from datetime import datetime
from typing import Any, Dict, List, Optional

from fetch_retry import fetch_com_retry


class PncpPaginacaoParaJson:
    """
    Responsável por:
    - percorrer as páginas da API PNCP
    - acumular todos os registros
    - remover duplicados
    - salvar tudo em um arquivo JSON
    - reaproveitar o JSON anterior fazendo append lógico
    - executar a coleta para uma ou várias modalidades
    - gerar resumo e agrupamento por modalidade
    """

    def __init__(
        self,
        base_url: str = "https://pncp.gov.br/api/consulta",
        endpoint: str = "/v1/contratacoes/proposta",
        codigo_modalidade: Optional[int] = 6,
        tamanho_pagina: int = 50,
        data_final: Optional[int] = None,
        pagina_inicial: int = 1,
        max_paginas: Optional[int] = None,
    ) -> None:
        self.base_url = base_url
        self.endpoint = endpoint
        self.url_final = f"{self.base_url}{self.endpoint}"
        self.codigo_modalidade = codigo_modalidade
        self.tamanho_pagina = tamanho_pagina
        self.data_final = data_final or int(datetime.now().strftime("%Y%m%d"))
        self.pagina_inicial = pagina_inicial
        self.max_paginas = max_paginas

        self.total_paginas_processadas = 0
        self.modalidades_processadas: List[int] = []

    def _montar_params(self, pagina: int) -> Dict[str, Any]:
        return {
            "dataFinal": self.data_final,
            "codigoModalidadeContratacao": self.codigo_modalidade,
            "pagina": pagina,
            "tamanhoPagina": self.tamanho_pagina,
        }

    def _extrair_registros_da_resposta(self, resposta: Any) -> List[Dict[str, Any]]:
        if isinstance(resposta, list):
            return resposta

        if isinstance(resposta, tuple):
            if len(resposta) > 0 and isinstance(resposta[0], list):
                return resposta[0]

        raise TypeError(
            f"Formato inesperado retornado por fetch_com_retry: "
            f"{type(resposta)} | valor: {repr(resposta)[:500]}"
        )

    def buscar_todas_as_paginas(self) -> List[Dict[str, Any]]:
        pagina = self.pagina_inicial
        todos_os_registros: List[Dict[str, Any]] = []
        total_paginas_conhecido = None
        ultima_pagina_processada = 0

        while True:
            if self.max_paginas is not None and pagina > self.max_paginas:
                print(f"Limite de páginas atingido: {self.max_paginas}")
                break

            params = self._montar_params(pagina)

            if total_paginas_conhecido is not None:
                print(f"\nConsultando página {pagina} de {total_paginas_conhecido}...")
            else:
                print(f"\nConsultando página {pagina}...")

            resposta_bruta = fetch_com_retry(self.url_final, params)
            registros = self._extrair_registros_da_resposta(resposta_bruta)

            if not registros:
                print(f"Nenhum registro encontrado na página {pagina}. Encerrando.")
                break

            quantidade = len(registros)
            todos_os_registros.extend(registros)
            ultima_pagina_processada = pagina

            print(f"Modalidade atual: {self.codigo_modalidade}")
            print(f"Página atual: {pagina}")
            print(f"Registros retornados nesta página: {quantidade}")
            print(f"Total acumulado até agora: {len(todos_os_registros)}")

            if quantidade < self.tamanho_pagina:
                total_paginas_conhecido = pagina
                print(f"Última página detectada: {pagina}")
                print(f"Total de páginas identificadas: {total_paginas_conhecido}")
                break

            pagina += 1

        print(f"\nColeta finalizada para modalidade {self.codigo_modalidade}.")
        print(f"Total de registros acumulados: {len(todos_os_registros)}")

        if total_paginas_conhecido is not None:
            print(f"Total de páginas processadas: {total_paginas_conhecido}")
            self.total_paginas_processadas = total_paginas_conhecido
        else:
            print(f"Total de páginas processadas: {ultima_pagina_processada}")
            self.total_paginas_processadas = ultima_pagina_processada

        return todos_os_registros

    def remover_duplicados(self, dados: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        vistos = set()
        unicos = []
        duplicados = 0

        for item in dados:
            chave = item.get("numeroControlePNCP")

            if chave:
                if chave not in vistos:
                    vistos.add(chave)
                    unicos.append(item)
                else:
                    duplicados += 1
            else:
                unicos.append(item)

        print(f"Total antes da deduplicação: {len(dados)}")
        print(f"Duplicados removidos: {duplicados}")
        print(f"Total após deduplicação: {len(unicos)}")

        return unicos

    def carregar_json_existente(self, caminho_arquivo: str) -> List[Dict[str, Any]]:
        if not os.path.exists(caminho_arquivo):
            print("Arquivo JSON ainda não existe. Será criado um novo.")
            return []

        try:
            with open(caminho_arquivo, "r", encoding="utf-8") as f:
                conteudo = json.load(f)

            if isinstance(conteudo, dict):
                if "dados" in conteudo and isinstance(conteudo["dados"], list):
                    print(f"JSON existente carregado com {len(conteudo['dados'])} registros.")
                    return conteudo["dados"]

                if "modalidades" in conteudo and isinstance(conteudo["modalidades"], dict):
                    dados_flat: List[Dict[str, Any]] = []
                    for grupo in conteudo["modalidades"].values():
                        if isinstance(grupo, dict) and isinstance(grupo.get("dados"), list):
                            dados_flat.extend(grupo["dados"])
                    print(f"JSON existente carregado com {len(dados_flat)} registros.")
                    return dados_flat

            if isinstance(conteudo, list):
                print(f"JSON existente carregado com {len(conteudo)} registros.")
                return conteudo

            print("JSON existente em formato inesperado. Será considerado vazio.")
            return []

        except json.JSONDecodeError:
            print("JSON existente está inválido ou vazio. Será considerado vazio.")
            return []

    def gerar_resumo_modalidades(self, dados: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        contagem = {}

        for item in dados:
            modalidade_id = item.get("modalidadeId")
            modalidade_nome = item.get("modalidadeNome")

            if modalidade_id is None or modalidade_nome is None:
                continue

            chave = (modalidade_id, modalidade_nome)
            contagem[chave] = contagem.get(chave, 0) + 1

        resumo = [
            {
                "modalidadeId": modalidade_id,
                "modalidadeNome": modalidade_nome,
                "quantidade": quantidade,
            }
            for (modalidade_id, modalidade_nome), quantidade in contagem.items()
        ]

        resumo.sort(key=lambda x: x["modalidadeId"])
        return resumo

    def agrupar_por_modalidade(self, dados: List[Dict[str, Any]]) -> Dict[str, Any]:
        grupos: Dict[str, Any] = {}

        for item in dados:
            modalidade_id = item.get("modalidadeId")
            modalidade_nome = item.get("modalidadeNome")

            if modalidade_id is None or modalidade_nome is None:
                continue

            chave = modalidade_nome

            if chave not in grupos:
                grupos[chave] = {
                    "modalidadeId": modalidade_id,
                    "codigo_modalidade": str(modalidade_id),
                    "quantidade": 0,
                    "dados": [],
                }

            grupos[chave]["dados"].append(item)
            grupos[chave]["quantidade"] += 1

        grupos_ordenados = dict(
            sorted(grupos.items(), key=lambda x: x[1]["modalidadeId"])
        )

        return grupos_ordenados

    def salvar_em_json(self, dados: List[Dict[str, Any]], caminho_arquivo: str) -> None:
        resumo_modalidades = self.gerar_resumo_modalidades(dados)
        modalidades_agrupadas = self.agrupar_por_modalidade(dados)

        print(f"Total de registros: {len(dados)}")
        print(f"Total de páginas: {self.total_paginas_processadas}")

        for item in resumo_modalidades:
            print(
                f'{item["modalidadeId"]} - {item["modalidadeNome"]}: '
                f'{item["quantidade"]} registro(s)'
            )

        print(f"Total de modalidades: {len(resumo_modalidades)}")

        payload = {
            "metadata": {
                "url": self.url_final,
                "codigo_modalidade": self.codigo_modalidade,
                "modalidades_processadas": self.modalidades_processadas,
                "data_final": self.data_final,
                "tamanho_pagina": self.tamanho_pagina,
                "totalRegistros": len(dados),
                "totalPaginas": self.total_paginas_processadas,
                "totalModalidades": len(resumo_modalidades),
                "ultima_atualizacao": datetime.now().isoformat(),
            },
            "resumoModalidades": resumo_modalidades,
            "modalidades": modalidades_agrupadas,
        }

        with open(caminho_arquivo, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=4, ensure_ascii=False)

        print(f"Arquivo JSON salvo com sucesso em: {caminho_arquivo}")

    def executar(self, caminho_arquivo: str = "pncp_contratacoes_proposta.json") -> List[Dict[str, Any]]:
        dados_existentes = self.carregar_json_existente(caminho_arquivo)
        dados_novos = self.buscar_todas_as_paginas()

        if self.codigo_modalidade is not None:
            self.modalidades_processadas = [self.codigo_modalidade]

        print(f"Registros já existentes no arquivo: {len(dados_existentes)}")
        print(f"Registros coletados nesta execução: {len(dados_novos)}")

        dados_combinados = dados_existentes + dados_novos
        dados_finais = self.remover_duplicados(dados_combinados)

        self.salvar_em_json(dados_finais, caminho_arquivo)
        return dados_finais

    def executar_varias_modalidades(
        self,
        caminho_arquivo: str = "pncp_contratacoes_todas_modalidades.json",
        modalidade_inicial: int = 1,
        modalidade_final: int = 13,
    ) -> List[Dict[str, Any]]:
        dados_finais: List[Dict[str, Any]] = []
        total_paginas_geral = 0
        modalidades_processadas: List[int] = []

        for codigo_modalidade in range(modalidade_inicial, modalidade_final + 1):
            print("\n" + "=" * 60)
            print(f"INICIANDO COLETA DA MODALIDADE {codigo_modalidade}")
            print("=" * 60)

            coletor = PncpPaginacaoParaJson(
                base_url=self.base_url,
                endpoint=self.endpoint,
                codigo_modalidade=codigo_modalidade,
                tamanho_pagina=self.tamanho_pagina,
                data_final=self.data_final,
                pagina_inicial=self.pagina_inicial,
                max_paginas=self.max_paginas,
            )

            dados_finais = coletor.executar(caminho_arquivo)
            total_paginas_geral += coletor.total_paginas_processadas
            modalidades_processadas.append(codigo_modalidade)

            print(f"FINALIZADA MODALIDADE {codigo_modalidade}")
            print(f"Total final de registros únicos no arquivo: {len(dados_finais)}")

        self.codigo_modalidade = None
        self.modalidades_processadas = modalidades_processadas
        self.total_paginas_processadas = total_paginas_geral

        self.salvar_em_json(dados_finais, caminho_arquivo)

        return dados_finais

    def listar_modalidades_do_json(self, caminho_arquivo: str) -> List[Dict[str, Any]]:
        """
        Lê o JSON salvo e imprime:
        - modalidadeId
        - modalidadeNome
        - quantidade de registros por modalidade
        """
        if not os.path.exists(caminho_arquivo):
            print(f"Arquivo não encontrado: {caminho_arquivo}")
            return []

        try:
            with open(caminho_arquivo, "r", encoding="utf-8") as f:
                conteudo = json.load(f)

            if isinstance(conteudo, dict) and "resumoModalidades" in conteudo:
                resumo = conteudo["resumoModalidades"]
            else:
                print("Formato de JSON inválido.")
                return []

            for item in resumo:
                print(
                    f'{item["modalidadeId"]} - {item["modalidadeNome"]}: '
                    f'{item["quantidade"]} registro(s)'
                )

            print(f"Total de modalidades: {len(resumo)}")
            return resumo

        except json.JSONDecodeError:
            print("Erro ao ler o JSON.")
            return []

In [52]:
if __name__ == "__main__":
    coletor = PncpPaginacaoParaJson(tamanho_pagina=50)

    dados = coletor.executar_varias_modalidades(
        caminho_arquivo="pncp_contratacoes_todas_modalidades.json",
        modalidade_inicial=1,
        modalidade_final=13,
    )

    print(f"\nTotal final consolidado: {len(dados)}")


INICIANDO COLETA DA MODALIDADE 1
JSON existente carregado com 105 registros.

Consultando página 1...
Status 204 (No Content). Fim dos dados.
Nenhum registro encontrado na página 1. Encerrando.

Coleta finalizada para modalidade 1.
Total de registros acumulados: 0
Total de páginas processadas: 0
Registros já existentes no arquivo: 105
Registros coletados nesta execução: 0
Total antes da deduplicação: 105
Duplicados removidos: 0
Total após deduplicação: 105
Total de registros: 105
Total de páginas: 0
4 - Concorrência - Eletrônica: 7 registro(s)
5 - Concorrência - Presencial: 2 registro(s)
6 - Pregão - Eletrônico: 13 registro(s)
7 - Pregão - Presencial: 2 registro(s)
8 - Dispensa: 66 registro(s)
10 - Manifestação de Interesse: 1 registro(s)
12 - Credenciamento: 14 registro(s)
Total de modalidades: 7
Arquivo JSON salvo com sucesso em: pncp_contratacoes_todas_modalidades.json
FINALIZADA MODALIDADE 1
Total final de registros únicos no arquivo: 105

INICIANDO COLETA DA MODALIDADE 2
JSON exi